In [ ]:
import torch
import gc
import random
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from datasets import load_dataset
from datasets import Dataset

import torch.nn.functional as F
from torch.utils.data import DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)
from transformers import AutoConfig


In [10]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [11]:
dataset = load_dataset("wangrongsheng/ag_news")

In [12]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MAX_MODEL_LENGTH = 2048
MAX_LENGTH = 2048

In [ ]:
label_token_dict = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Science"
}


def build_prompt(text):

    prompt = f"""Article: {text}

Theme:"""

    return prompt


def build_full_text(text, label):

    theme = label_token_dict[label]

    full_text = f"""Article: {text}

Theme: {theme}"""

    return full_text


def preprocess_dataset(dataset, tokenizer):
    records = dataset.to_dict("records")

    reviews    = [r["text"]  for r in records]
    raw_labels = [r["label"] for r in records]

    full_texts   = [build_full_text(rev, lbl) for rev, lbl in zip(reviews, raw_labels)]
    prompt_texts = [build_prompt(rev) for rev in reviews]

    full_enc = tokenizer(
        full_texts,
        add_special_tokens=False,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
    )
    prompt_enc = tokenizer(
        prompt_texts,
        add_special_tokens=False,
    )

    labels_list = [
        [
            -100 if (i < len(prompt_ids) or attn == 0) else tok
            for i, (tok, attn) in enumerate(zip(input_ids, attention_mask))
        ]
        for input_ids, attention_mask, prompt_ids in zip(
            full_enc["input_ids"],
            full_enc["attention_mask"],
            prompt_enc["input_ids"],
        )
    ]

    true_labels_list = [label_token_dict[l] for l in raw_labels]

    return Dataset.from_dict({
        "input_ids":      full_enc["input_ids"],
        "attention_mask": full_enc["attention_mask"],
        "labels":         labels_list,
        "true_label":     true_labels_list,
        "texts":          reviews,
    })


In [14]:
# Model tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [15]:
def fits_model(example):

    text = build_full_text(
        example["text"],
        example["label"]
    )

    tokens = tokenizer(
        text,
        add_special_tokens=False
    )["input_ids"]

    return len(tokens) <= MAX_MODEL_LENGTH

In [ ]:
import pandas as pd
import random

BATCH_SIZE      = 1
N_PER_CLASS     = 50  # 50 x 4 classes = 200 total
AGNEWS_CLASSES  = ["World", "Sports", "Business", "Science"]

test_raw      = pd.DataFrame(dataset["test"])
filtered_test = test_raw[test_raw.apply(fits_model, axis=1)].reset_index(drop=True)
test_dataset  = preprocess_dataset(filtered_test, tokenizer)

random.seed(42)
eval_idx = []
for lbl in AGNEWS_CLASSES:
    idx = [i for i in range(len(test_dataset)) if test_dataset[i]["true_label"] == lbl]
    eval_idx.extend(random.sample(idx, N_PER_CLASS))

test_subset = test_dataset.select(eval_idx)
test_subset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_loader = DataLoader(test_subset, batch_size=BATCH_SIZE, shuffle=False)

eval_texts  = [test_subset[i]["texts"]      for i in range(len(test_subset))]
eval_labels = [test_subset[i]["true_label"] for i in range(len(test_subset))]
print(f"Eval: {len(eval_idx)} examples | "
      + ", ".join(f"{l}: {eval_labels.count(l)}" for l in AGNEWS_CLASSES))


In [ ]:
CHECKPOINT_DIR = "../checkpoints/agnews"

config_11 = AutoConfig.from_pretrained(MODEL_NAME, local_files_only=True)
config_11.num_hidden_layers = 11


def load_model(n_layers, ckpt_name):
    if n_layers == 22:
        m = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME, local_files_only=True, attn_implementation="eager"
        )
    else:
        m = AutoModelForCausalLM.from_config(config_11, attn_implementation="eager")
    m.load_state_dict(torch.load(f"{CHECKPOINT_DIR}/{ckpt_name}", map_location="cpu"))
    m.eval()
    return m


MODELS = {
    "teacher":       load_model(22, "best_teacher_model.pt"),
    "baseline":      load_model(11, "best_baseline_model.pt"),
    "bad_student":   load_model(11, "best_bad_student_model.pt"),
    "student":       load_model(11, "best_student_model.pt"),
    "student_local": load_model(11, "best_student_local_model.pt"),
}
MODEL_LABELS = {
    "teacher":       "P (teacher)",
    "baseline":      "B (no KD)",
    "bad_student":   "S_bad",
    "student":       "S1",
    "student_local": "S2",
}
print("Loaded:", list(MODELS.keys()))


# Métricas de comparación

In [ ]:
def frobenius_difference(
    x: torch.Tensor,
    y: torch.Tensor
):
    return torch.norm(
        x - y,
        p="fro"
    )


def cosine_similarity_tensor(
    x: torch.Tensor,
    y: torch.Tensor,
    eps: float = 1e-8
):
    x_flat = x.reshape(-1)
    y_flat = y.reshape(-1)

    similarity = F.cosine_similarity(
        x_flat.unsqueeze(0),
        y_flat.unsqueeze(0),
        dim=1,
        eps=eps
    )

    return similarity.squeeze()

def js_divergence_attention(
    attn_teacher: torch.Tensor,
    attn_student: torch.Tensor,
    eps: float = 1e-8
):
    # normalizar filas
    p = attn_teacher / (
        attn_teacher.sum(dim=-1, keepdim=True)
        + eps
    )

    q = attn_student / (
        attn_student.sum(dim=-1, keepdim=True)
        + eps
    )

    m = 0.5 * (p + q)

    kl_pm = torch.sum(
        p * torch.log(
            (p + eps) / (m + eps)
        ),
        dim=-1
    )

    kl_qm = torch.sum(
        q * torch.log(
            (q + eps) / (m + eps)
        ),
        dim=-1
    )

    jsd_rows = 0.5 * (
        kl_pm + kl_qm
    )

    return jsd_rows.mean()

def linear_cka(X, Y):
    """Linear CKA between activation matrices X:[n,d1] and Y:[n,d2]."""
    X = X.float();  Y = Y.float()
    n = X.shape[0]
    X = X - X.mean(0, keepdim=True)
    Y = Y - Y.mean(0, keepdim=True)
    K = X @ X.T;  L = Y @ Y.T
    H = torch.eye(n, device=X.device) - torch.ones(n, n, device=X.device) / n
    Kc = H @ K @ H;  Lc = H @ L @ H
    hsic_kl = (Kc * Lc).sum()
    hsic_kk = (Kc * Kc).sum()
    hsic_ll = (Lc * Lc).sum()
    return (hsic_kl / (torch.sqrt(hsic_kk * hsic_ll) + 1e-10)).item()


# Rango efectivo

In [19]:
def effective_rank_participation_ratio(
    x: torch.Tensor,
    eps: float = 1e-12
):

    x = x.float()

    singular_values = (
        torch.linalg.svdvals(x)
    )

    power = singular_values**2

    numerator = (
        power.sum()**2
    )

    denominator = (
        (power**2).sum()
        + eps
    )

    rank_eff = (
        numerator
        / denominator
    )

    return rank_eff


def effective_rank_entropy(
    x: torch.Tensor,
    eps: float = 1e-12
):

    x = x.float()

    singular_values = (
        torch.linalg.svdvals(x)
    )

    p = singular_values / (
        singular_values.sum()
        + eps
    )

    entropy = -torch.sum(
        p * torch.log(p + eps)
    )

    rank_eff = torch.exp(entropy)

    return rank_eff

# Atención promedio

In [20]:
def compute_average_attentions(
    model,
    dataloader,
    device,
    max_batches=None,
    print_every=50
):

    model.eval()

    n_layers = (
        model.config.num_hidden_layers
    )

    attention_sums = [

        torch.zeros(
            (2048, 2048),
            dtype=torch.float32,
            device="cpu"
        )

        for _ in range(n_layers)
    ]

    n_examples = 0

    with torch.no_grad():

        for batch_idx, batch in enumerate(
            dataloader
        ):

            if (
                max_batches is not None
                and batch_idx >= max_batches
            ):
                break

            input_ids = batch[
                "input_ids"
            ].to(device)

            attention_mask = batch[
                "attention_mask"
            ].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_attentions=True
            )

            attentions = (
                outputs.attentions
            )

            batch_size = (
                input_ids.size(0)
            )

            for layer_idx in range(
                n_layers
            ):

                attn = attentions[
                    layer_idx
                ]

                # (B, H, S, S)
                # promedio heads
                attn = attn.mean(
                    dim=1
                )

                # (B, S, S)
                attn = (
                    attn
                    .float()
                    .cpu()
                )

                # suma batch
                attn_sum = attn.sum(
                    dim=0
                )

                attention_sums[
                    layer_idx
                ] += attn_sum

            n_examples += batch_size

            if (
                batch_idx
                % print_every
                == 0
            ):

                print(
                    f"Batch "
                    f"{batch_idx} | "
                    f"Examples "
                    f"{n_examples}"
                )

            del (
                input_ids,
                attention_mask,
                outputs,
                attentions,
                attn
            )

            gc.collect()

            torch.cuda.empty_cache()

    attention_means = [

        attn_sum / n_examples

        for attn_sum
        in attention_sums
    ]

    return attention_means

In [ ]:
def compute_hidden_states(model, prompts, tokenizer, device, n=100, max_seq=512):
    """
    Returns [n_layers+1, n, hidden_dim] – last real-token hidden vector per layer.
    """
    model.eval()
    all_hidden = []
    with torch.no_grad():
        for prompt in prompts[:n]:
            enc = tokenizer(prompt, return_tensors="pt", truncation=True,
                            max_length=max_seq, add_special_tokens=False)
            input_ids = enc["input_ids"].to(device)
            attn_mask = enc["attention_mask"].to(device)
            last_idx  = int(attn_mask[0].nonzero()[-1].item())
            out = model(input_ids=input_ids, attention_mask=attn_mask,
                        output_hidden_states=True)
            hidden = torch.stack(
                [h[0, last_idx, :] for h in out.hidden_states]
            ).cpu()  # [n_layers+1, hidden_dim]
            all_hidden.append(hidden)
            del out;  torch.cuda.empty_cache()
    return torch.stack(all_hidden, dim=1)  # [n_layers+1, n, hidden_dim]


def compute_head_avg_patterns(model, prompts, tokenizer, device, n=50, max_seq=128):
    """
    Returns [n_layers, n_heads, max_seq] – avg attention from last token per head.
    """
    model.eval()
    n_layers = model.config.num_hidden_layers
    n_heads  = model.config.num_attention_heads
    sums  = torch.zeros(n_layers, n_heads, max_seq, dtype=torch.float32)
    count = 0
    with torch.no_grad():
        for prompt in prompts[:n]:
            enc = tokenizer(prompt, return_tensors="pt", truncation=True,
                            max_length=max_seq, add_special_tokens=False)
            input_ids = enc["input_ids"].to(device)
            attn_mask = enc["attention_mask"].to(device)
            seq_len   = input_ids.size(1)
            last_idx  = int(attn_mask[0].nonzero()[-1].item())
            out = model(input_ids=input_ids, attention_mask=attn_mask,
                        output_attentions=True)
            for l in range(n_layers):
                # [1, n_heads, S, S] -> [n_heads, seq_len]
                ha = out.attentions[l][0, :, last_idx, :seq_len].float().cpu()
                padded = torch.zeros(n_heads, max_seq)
                padded[:, :seq_len] = ha
                sums[l] += padded
            count += 1
            del out;  torch.cuda.empty_cache()
    return sums / count  # [n_layers, n_heads, max_seq]


## Teacher

In [ ]:
# Central attention computation for all 5 models
model_attentions = {}
for name, model in MODELS.items():
    print(f"Computing attentions for {MODEL_LABELS[name]}...")
    model.to(device)
    model_attentions[name] = compute_average_attentions(model, test_loader, device)
    model.cpu()
    torch.cuda.empty_cache()
    gc.collect()

# 22-layer teacher attentions (needed by teacher_avg_attentions + teacher_rollouts)
teacher_attentions = model_attentions["teacher"]

# Aliases — existing pairwise comparison cells use these names unchanged
baseline_attentions      = model_attentions["baseline"]
bad_student_attentions   = model_attentions["bad_student"]
student_attentions       = model_attentions["student"]
student_local_attentions = model_attentions["student_local"]

print("All attention maps computed.")


In [25]:
teacher_avg_attentions = [
    (teacher_attentions[2*i] + teacher_attentions[2*i + 1]) / 2
    for i in range(11)
]

In [26]:
def rollout_pair(A1, A2):
    I = torch.eye(A1.shape[-1], device=A1.device)
    
    # agregar residual y renormalizar
    A1 = A1 + I
    A1 = A1 / A1.sum(dim=-1, keepdim=True)
    
    A2 = A2 + I
    A2 = A2 / A2.sum(dim=-1, keepdim=True)
    
    return A1 @ A2

teacher_rollouts = [rollout_pair(teacher_attentions[2*i], teacher_attentions[2*i+1]) for i in range(11)]

# Teacher - Baseline comparison

In [31]:
# Cosine similarity entre attention maps
print("Similitud coseno con promedio de atenciones")
for i, (battn, tattn) in enumerate(zip(baseline_attentions, teacher_avg_attentions)):
    print(cosine_similarity_tensor(battn, tattn))

print("\n")

print("Similitud coseno con Attention Rollout")
for i, (battn, tattn) in enumerate(zip(baseline_attentions, teacher_rollouts)):
    print(cosine_similarity_tensor(battn, tattn))

Similitud coseno con promedio de atenciones
tensor(0.9824)
tensor(0.3706)
tensor(0.4279)
tensor(0.4377)
tensor(0.5215)
tensor(0.3354)
tensor(0.3028)
tensor(0.2759)
tensor(0.4222)
tensor(0.2553)
tensor(0.3158)


Similitud coseno con Attention Rollout
tensor(0.3634)
tensor(0.2957)
tensor(0.4047)
tensor(0.4138)
tensor(0.4496)
tensor(0.2799)
tensor(0.2395)
tensor(0.2164)
tensor(0.3411)
tensor(0.2010)
tensor(0.2565)


In [32]:
# Frobenius distance entre attention maps
print("Distancia Frobenius con promedio de atenciones")
for i, (battn, tattn) in enumerate(zip(baseline_attentions, teacher_avg_attentions)):
    print(frobenius_difference(battn, tattn))

print("\n")

print("Distancia Frobenius con Attention Rollout")
for i, (battn, tattn) in enumerate(zip(baseline_attentions, teacher_rollouts)):
    print(frobenius_difference(battn, tattn))

Distancia Frobenius con promedio de atenciones
tensor(1.0457)
tensor(19.9522)
tensor(39.1852)
tensor(37.7777)
tensor(21.3202)
tensor(21.1765)
tensor(18.1935)
tensor(17.9629)
tensor(17.2216)
tensor(18.9041)
tensor(19.2287)


Distancia Frobenius con Attention Rollout
tensor(11.4308)
tensor(22.2240)
tensor(31.8282)
tensor(31.0298)
tensor(22.1431)
tensor(22.4927)
tensor(20.4605)
tensor(20.5095)
tensor(20.0608)
tensor(21.4267)
tensor(21.1837)


In [33]:
# JS local
print("JS promediada por fila con promedio de atenciones")
for i, (battn, tattn) in enumerate(zip(baseline_attentions, teacher_avg_attentions)):
    print(js_divergence_attention(battn, tattn))

print("\n")

print("JS promediada por fila con Attention Rollout")
for i, (battn, tattn) in enumerate(zip(baseline_attentions, teacher_rollouts)):
    print(js_divergence_attention(battn, tattn))

JS promediada por fila con promedio de atenciones
tensor(0.0045)
tensor(0.1700)
tensor(0.4478)
tensor(0.4149)
tensor(0.2379)
tensor(0.2811)
tensor(0.2568)
tensor(0.2941)
tensor(0.2650)
tensor(0.3232)
tensor(0.2817)


JS promediada por fila con Attention Rollout
tensor(0.0995)
tensor(0.2971)
tensor(0.5041)
tensor(0.4792)
tensor(0.3293)
tensor(0.3624)
tensor(0.3323)
tensor(0.3617)
tensor(0.3395)
tensor(0.3921)
tensor(0.3543)


In [34]:
# Participation ratio
print("Participation ratio")
print("index | student | teacher_avg | teacher_rollout")
print("-----------------------------------------------")
for i, (battn, tattn, tattn2) in enumerate(zip(baseline_attentions, teacher_avg_attentions, teacher_rollouts)):
    print(f"{i} | {effective_rank_participation_ratio(battn):4f} | {effective_rank_participation_ratio(tattn):4f} | {effective_rank_participation_ratio(tattn2):4f}")

Participation ratio
index | student | teacher_avg | teacher_rollout
-----------------------------------------------
0 | 1.275496 | 1.356875 | 67.457176
1 | 1.605922 | 1.005610 | 1.724615
2 | 1.597238 | 1.000082 | 1.265335
3 | 1.883908 | 1.000213 | 1.277876
4 | 1.836384 | 1.005978 | 1.622420
5 | 2.638350 | 1.002858 | 1.704638
6 | 2.376256 | 1.006170 | 1.995486
7 | 3.335663 | 1.007666 | 2.028602
8 | 2.559661 | 1.007667 | 1.945815
9 | 4.253060 | 1.007110 | 1.909445
10 | 3.194835 | 1.006790 | 1.885484


In [35]:
# Entropy based effective rank
print("Entropy based effective rank")
print("index | student | teacher_avg | teacher_rollout")
print("-----------------------------------------------")
for i, (battn, tattn, tattn2) in enumerate(zip(baseline_attentions, teacher_avg_attentions, teacher_rollouts)):
    print(f"{i} | {effective_rank_entropy(battn):4f} | {effective_rank_entropy(tattn):4f} | {effective_rank_entropy(tattn2):4f}")

Entropy based effective rank
index | student | teacher_avg | teacher_rollout
-----------------------------------------------
0 | 11.610521 | 22.324389 | 2010.146240
1 | 28.762945 | 4.120300 | 1798.869263
2 | 27.222998 | 1.356920 | 1633.819702
3 | 37.570496 | 1.611693 | 1643.186646
4 | 39.992477 | 2.645257 | 1778.611450
5 | 49.555668 | 2.623758 | 1796.103394
6 | 55.765526 | 4.044635 | 1836.495728
7 | 58.389290 | 4.690934 | 1839.722046
8 | 58.099644 | 4.206147 | 1830.746948
9 | 58.180458 | 3.517950 | 1826.503052
10 | 62.297825 | 5.958082 | 1822.801880


# Teacher - Bad Student comparison

In [40]:
# Cosine similarity entre attention maps
print("Similitud coseno con promedio de atenciones")
for i, (bsattn, tattn) in enumerate(zip(bad_student_attentions, teacher_avg_attentions)):
    print(cosine_similarity_tensor(bsattn, tattn))

print("\n")

print("Similitud coseno con Attention Rollout")
for i, (bsattn, tattn) in enumerate(zip(bad_student_attentions, teacher_rollouts)):
    print(cosine_similarity_tensor(bsattn, tattn))

Similitud coseno con promedio de atenciones
tensor(0.9674)
tensor(0.6984)
tensor(0.3183)
tensor(0.3922)
tensor(0.3470)
tensor(0.3997)
tensor(0.3437)
tensor(0.3468)
tensor(0.2811)
tensor(0.4006)
tensor(0.2811)


Similitud coseno con Attention Rollout
tensor(0.3679)
tensor(0.5865)
tensor(0.3033)
tensor(0.3737)
tensor(0.2975)
tensor(0.3373)
tensor(0.2736)
tensor(0.2720)
tensor(0.2168)
tensor(0.3182)
tensor(0.2156)


In [41]:
# Frobenius distance entre attention maps
print("Distancia Frobenius con promedio de atenciones")
for i, (bsattn, tattn) in enumerate(zip(bad_student_attentions, teacher_avg_attentions)):
    print(frobenius_difference(bsattn, tattn))

print("\n")

print("Distancia Frobenius con Attention Rollout")
for i, (bsattn, tattn) in enumerate(zip(bad_student_attentions, teacher_rollouts)):
    print(frobenius_difference(bsattn, tattn))

Distancia Frobenius con promedio de atenciones
tensor(1.4239)
tensor(17.1706)
tensor(40.1491)
tensor(38.5048)
tensor(22.9013)
tensor(20.8623)
tensor(17.9269)
tensor(17.4185)
tensor(18.2144)
tensor(17.8162)
tensor(19.4310)


Distancia Frobenius con Attention Rollout
tensor(11.4060)
tensor(19.9073)
tensor(32.6832)
tensor(31.6298)
tensor(23.4206)
tensor(22.1576)
tensor(20.2026)
tensor(20.0316)
tensor(20.8673)
tensor(20.4940)
tensor(21.3683)


In [42]:
# JS local
print("JS promediada por fila con promedio de atenciones")
for i, (bsattn, tattn) in enumerate(zip(bad_student_attentions, teacher_avg_attentions)):
    print(js_divergence_attention(bsattn, tattn))

print("\n")

print("JS promediada por fila con Attention Rollout")
for i, (bsattn, tattn) in enumerate(zip(bad_student_attentions, teacher_rollouts)):
    print(js_divergence_attention(bsattn, tattn))

JS promediada por fila con promedio de atenciones
tensor(0.0054)
tensor(0.1046)
tensor(0.4781)
tensor(0.4304)
tensor(0.2159)
tensor(0.2093)
tensor(0.2001)
tensor(0.2049)
tensor(0.2379)
tensor(0.2180)
tensor(0.2111)


JS promediada por fila con Attention Rollout
tensor(0.0993)
tensor(0.2360)
tensor(0.5333)
tensor(0.4962)
tensor(0.3267)
tensor(0.3116)
tensor(0.2909)
tensor(0.2951)
tensor(0.3275)
tensor(0.3104)
tensor(0.3117)


In [43]:
# Participation ratio
print("Participation ratio")
print("index | student | teacher_avg | teacher_rollout")
print("-----------------------------------------------")
for i, (bsattn, tattn, tattn2) in enumerate(zip(bad_student_attentions, teacher_avg_attentions, teacher_rollouts)):
    print(f"{i} | {effective_rank_participation_ratio(bsattn):4f} | {effective_rank_participation_ratio(tattn):4f} | {effective_rank_participation_ratio(tattn2):4f}")

Participation ratio
index | student | teacher_avg | teacher_rollout
-----------------------------------------------
0 | 1.291532 | 1.356875 | 67.457176
1 | 1.309881 | 1.005610 | 1.724615
2 | 1.426726 | 1.000082 | 1.265335
3 | 1.358773 | 1.000213 | 1.277876
4 | 1.360982 | 1.005978 | 1.622420
5 | 1.398054 | 1.002858 | 1.704638
6 | 1.455059 | 1.006170 | 1.995486
7 | 1.519558 | 1.007666 | 2.028602
8 | 1.465775 | 1.007667 | 1.945815
9 | 1.435512 | 1.007110 | 1.909445
10 | 1.545660 | 1.006790 | 1.885484


In [44]:
# Entropy based effective rank
print("Entropy based effective rank")
print("index | student | teacher_avg | teacher_rollout")
print("-----------------------------------------------")
for i, (bsattn, tattn, tattn2) in enumerate(zip(bad_student_attentions, teacher_avg_attentions, teacher_rollouts)):
    print(f"{i} | {effective_rank_entropy(bsattn):4f} | {effective_rank_entropy(tattn):4f} | {effective_rank_entropy(tattn2):4f}")

Entropy based effective rank
index | student | teacher_avg | teacher_rollout
-----------------------------------------------
0 | 12.529155 | 22.324389 | 2010.146240
1 | 12.878530 | 4.120300 | 1798.869263
2 | 12.064932 | 1.356920 | 1633.819702
3 | 17.062733 | 1.611693 | 1643.186646
4 | 17.879768 | 2.645257 | 1778.611450
5 | 21.525631 | 2.623758 | 1796.103394
6 | 28.012730 | 4.044635 | 1836.495728
7 | 28.416212 | 4.690934 | 1839.722046
8 | 26.996763 | 4.206147 | 1830.746948
9 | 26.414501 | 3.517950 | 1826.503052
10 | 23.052923 | 5.958082 | 1822.801880


## Teacher - Student comparison

In [49]:
# Cosine similarity entre attention maps
print("Similitud coseno con promedio de atenciones")
for i, (sattn, tattn) in enumerate(zip(student_attentions, teacher_avg_attentions)):
    print(cosine_similarity_tensor(sattn, tattn))

print("\n")

print("Similitud coseno con Attention Rollout")
for i, (sattn, tattn) in enumerate(zip(student_attentions, teacher_rollouts)):
    print(cosine_similarity_tensor(sattn, tattn))

Similitud coseno con promedio de atenciones
tensor(0.9708)
tensor(0.3063)
tensor(0.2321)
tensor(0.2216)
tensor(0.2928)
tensor(0.2838)
tensor(0.3441)
tensor(0.3391)
tensor(0.2765)
tensor(0.3232)
tensor(0.2298)


Similitud coseno con Attention Rollout
tensor(0.3653)
tensor(0.2378)
tensor(0.2289)
tensor(0.2188)
tensor(0.2630)
tensor(0.2412)
tensor(0.2843)
tensor(0.2702)
tensor(0.2151)
tensor(0.2613)
tensor(0.1828)


In [50]:
# Frobenius distance entre attention maps
print("Distancia Frobenius con promedio de atenciones")
for i, (sattn, tattn) in enumerate(zip(student_attentions, teacher_avg_attentions)):
    print(frobenius_difference(sattn, tattn))

print("\n")

print("Distancia Frobenius con Attention Rollout")
for i, (sattn, tattn) in enumerate(zip(student_attentions, teacher_rollouts)):
    print(frobenius_difference(sattn, tattn))

Distancia Frobenius con promedio de atenciones
tensor(1.3403)
tensor(20.3864)
tensor(40.6362)
tensor(39.5156)
tensor(23.2047)
tensor(21.5605)
tensor(17.9137)
tensor(17.4644)
tensor(18.2383)
tensor(18.2939)
tensor(19.7375)


Distancia Frobenius con Attention Rollout
tensor(11.4171)
tensor(22.5916)
tensor(33.1117)
tensor(32.5615)
tensor(23.6135)
tensor(22.7287)
tensor(20.1373)
tensor(20.0419)
tensor(20.8694)
tensor(20.8386)
tensor(21.5595)


In [51]:
# JS local
print("JS promediada por fila con promedio de atenciones")
for i, (sattn, tattn) in enumerate(zip(student_attentions, teacher_avg_attentions)):
    print(js_divergence_attention(sattn, tattn))

print("\n")

print("JS promediada por fila con Attention Rollout")
for i, (sattn, tattn) in enumerate(zip(student_attentions, teacher_rollouts)):
    print(js_divergence_attention(sattn, tattn))

JS promediada por fila con promedio de atenciones
tensor(0.0044)
tensor(0.1721)
tensor(0.5019)
tensor(0.4746)
tensor(0.2089)
tensor(0.2086)
tensor(0.1856)
tensor(0.1964)
tensor(0.2244)
tensor(0.2239)
tensor(0.2340)


JS promediada por fila con Attention Rollout
tensor(0.1014)
tensor(0.3062)
tensor(0.5524)
tensor(0.5345)
tensor(0.3237)
tensor(0.3185)
tensor(0.2806)
tensor(0.2895)
tensor(0.3205)
tensor(0.3176)
tensor(0.3274)


## Teacher - Student local comparison

In [56]:
# Cosine similarity entre attention maps
print("Similitud coseno con promedio de atenciones")
for i, (sattn, tattn) in enumerate(zip(student_local_attentions, teacher_avg_attentions)):
    print(cosine_similarity_tensor(sattn, tattn))

print("\n")

print("Similitud coseno con Attention Rollout")
for i, (sattn, tattn) in enumerate(zip(student_local_attentions, teacher_rollouts)):
    print(cosine_similarity_tensor(sattn, tattn))

Similitud coseno con promedio de atenciones
tensor(0.9783)
tensor(0.3770)
tensor(0.3693)
tensor(0.2314)
tensor(0.3501)
tensor(0.3006)
tensor(0.3036)
tensor(0.2821)
tensor(0.3170)
tensor(0.2977)
tensor(0.2681)


Similitud coseno con Attention Rollout
tensor(0.3646)
tensor(0.3025)
tensor(0.3638)
tensor(0.2352)
tensor(0.3255)
tensor(0.2691)
tensor(0.2612)
tensor(0.2278)
tensor(0.2499)
tensor(0.2503)
tensor(0.2202)


In [57]:
# Frobenius distance entre attention maps
print("Distancia Frobenius con promedio de atenciones")
for i, (sattn, tattn) in enumerate(zip(student_local_attentions, teacher_avg_attentions)):
    print(frobenius_difference(sattn, tattn))

print("\n")

print("Distancia Frobenius con Attention Rollout")
for i, (sattn, tattn) in enumerate(zip(student_local_attentions, teacher_rollouts)):
    print(frobenius_difference(sattn, tattn))

Distancia Frobenius con promedio de atenciones
tensor(1.1417)
tensor(19.9252)
tensor(39.6231)
tensor(39.4679)
tensor(22.8287)
tensor(21.4462)
tensor(18.1679)
tensor(17.8135)
tensor(17.9941)
tensor(18.4677)
tensor(19.5314)


Distancia Frobenius con Attention Rollout
tensor(11.4111)
tensor(22.1857)
tensor(32.1375)
tensor(32.4657)
tensor(23.2141)
tensor(22.5574)
tensor(20.2850)
tensor(20.3019)
tensor(20.6724)
tensor(20.9315)
tensor(21.3781)


In [58]:
# JS local
print("JS promediada por fila con promedio de atenciones")
for i, (sattn, tattn) in enumerate(zip(student_local_attentions, teacher_avg_attentions)):
    print(js_divergence_attention(sattn, tattn))

print("\n")

print("JS promediada por fila con Attention Rollout")
for i, (sattn, tattn) in enumerate(zip(student_local_attentions, teacher_rollouts)):
    print(js_divergence_attention(sattn, tattn))

JS promediada por fila con promedio de atenciones
tensor(0.0040)
tensor(0.1637)
tensor(0.4665)
tensor(0.4653)
tensor(0.2047)
tensor(0.2238)
tensor(0.2013)
tensor(0.2267)
tensor(0.2193)
tensor(0.2356)
tensor(0.2284)


JS promediada por fila con Attention Rollout
tensor(0.1039)
tensor(0.2959)
tensor(0.5183)
tensor(0.5245)
tensor(0.3144)
tensor(0.3228)
tensor(0.2909)
tensor(0.3117)
tensor(0.3130)
tensor(0.3236)
tensor(0.3194)


# Baseline - Student comparison

In [59]:
# Cosine similarity entre attention maps
print("Similitud coseno entre atenciones")
for i, (battn, sattn) in enumerate(zip(baseline_attentions, student_attentions)):
    print(cosine_similarity_tensor(battn, sattn))

Similitud coseno entre atenciones
tensor(0.9753)
tensor(0.7986)
tensor(0.7043)
tensor(0.6667)
tensor(0.6738)
tensor(0.6736)
tensor(0.7503)
tensor(0.6928)
tensor(0.7251)
tensor(0.6414)
tensor(0.6989)


In [60]:
# Frobenius distance entre attention maps
print("Distancia Frobenius entre atenciones")
for i, (battn, sattn) in enumerate(zip(baseline_attentions, student_attentions)):
    print(frobenius_difference(battn, sattn))

Distancia Frobenius entre atenciones
tensor(1.2422)
tensor(3.8840)
tensor(4.9986)
tensor(5.6689)
tensor(5.5546)
tensor(5.3805)
tensor(4.5084)
tensor(5.2839)
tensor(4.9391)
tensor(6.0396)
tensor(5.2164)


In [61]:
# JS local
print("JS promediada por fila entre atenciones")
for i, (battn, sattn) in enumerate(zip(baseline_attentions, student_attentions)):
    print(js_divergence_attention(battn, sattn))

JS promediada por fila entre atenciones
tensor(0.0049)
tensor(0.0590)
tensor(0.0951)
tensor(0.1042)
tensor(0.1003)
tensor(0.1100)
tensor(0.0750)
tensor(0.1024)
tensor(0.0941)
tensor(0.1304)
tensor(0.0990)


# Baseline - Student local comparison

In [62]:
# Cosine similarity entre attention maps
print("Similitud coseno entre atenciones")
for i, (battn, slattn) in enumerate(zip(baseline_attentions, student_local_attentions)):
    print(cosine_similarity_tensor(battn, slattn))

Similitud coseno entre atenciones
tensor(0.9823)
tensor(0.7789)
tensor(0.6693)
tensor(0.6569)
tensor(0.7037)
tensor(0.7059)
tensor(0.7326)
tensor(0.7002)
tensor(0.7249)
tensor(0.5924)
tensor(0.6686)


In [63]:
# Frobenius distance entre attention maps
print("Distancia Frobenius entre atenciones")
for i, (battn, slattn) in enumerate(zip(baseline_attentions, student_local_attentions)):
    print(frobenius_difference(battn, slattn))

Distancia Frobenius entre atenciones
tensor(1.0506)
tensor(4.1074)
tensor(5.5094)
tensor(5.7083)
tensor(5.3455)
tensor(5.1860)
tensor(4.7256)
tensor(5.2494)
tensor(4.9747)
tensor(6.5695)
tensor(5.6015)


In [64]:
# JS local
print("JS promediada por fila entre atenciones")
for i, (battn, slattn) in enumerate(zip(baseline_attentions, student_local_attentions)):
    print(js_divergence_attention(battn, slattn))


JS promediada por fila entre atenciones
tensor(0.0051)
tensor(0.0658)
tensor(0.0869)
tensor(0.0941)
tensor(0.0823)
tensor(0.0871)
tensor(0.0739)
tensor(0.1017)
tensor(0.0924)
tensor(0.1367)
tensor(0.0986)


# Baseline - Bad Student comparison

In [65]:
# Cosine similarity entre attention maps
print("Similitud coseno entre atenciones")
for i, (battn, bsattn) in enumerate(zip(baseline_attentions, bad_student_attentions)):
    print(cosine_similarity_tensor(battn, bsattn))

Similitud coseno entre atenciones
tensor(0.9741)
tensor(0.7352)
tensor(0.8371)
tensor(0.8124)
tensor(0.7998)
tensor(0.7667)
tensor(0.8106)
tensor(0.7259)
tensor(0.7408)
tensor(0.6683)
tensor(0.6753)


In [66]:
# Frobenius distance entre attention maps
print("Distancia Frobenius entre atenciones")
for i, (battn, bsattn) in enumerate(zip(baseline_attentions, bad_student_attentions)):
    print(frobenius_difference(battn, bsattn))

Distancia Frobenius entre atenciones
tensor(1.2758)
tensor(4.9730)
tensor(3.7279)
tensor(4.3195)
tensor(4.4785)
tensor(4.6107)
tensor(3.9307)
tensor(5.0146)
tensor(4.8223)
tensor(5.8260)
tensor(5.4088)


In [67]:
# JS local
print("JS promediada por fila entre atenciones")
for i, (battn, bsattn) in enumerate(zip(baseline_attentions, bad_student_attentions)):
    print(js_divergence_attention(battn, bsattn))

JS promediada por fila entre atenciones
tensor(0.0047)
tensor(0.0520)
tensor(0.0443)
tensor(0.0580)
tensor(0.0662)
tensor(0.0789)
tensor(0.0621)
tensor(0.0960)
tensor(0.0867)
tensor(0.1270)
tensor(0.1079)


# Bad Student - Student comparison

In [68]:
# Cosine similarity entre attention maps
print("Similitud coseno entre atenciones")
for i, (bsattn, sattn) in enumerate(zip(bad_student_attentions, student_attentions)):
    print(cosine_similarity_tensor(bsattn, sattn))

Similitud coseno entre atenciones
tensor(0.9837)
tensor(0.8042)
tensor(0.9012)
tensor(0.8995)
tensor(0.9114)
tensor(0.9095)
tensor(0.9431)
tensor(0.9368)
tensor(0.9140)
tensor(0.9192)
tensor(0.8771)


In [69]:
# Frobenius distance entre attention maps
print("Distancia Frobenius entre atenciones")
for i, (bsattn, sattn) in enumerate(zip(bad_student_attentions, student_attentions)):
    print(frobenius_difference(bsattn, sattn))

Distancia Frobenius entre atenciones
tensor(1.0117)
tensor(4.2813)
tensor(2.6474)
tensor(2.7373)
tensor(2.4127)
tensor(2.4339)
tensor(1.9518)
tensor(2.0314)
tensor(2.3936)
tensor(2.3419)
tensor(2.8789)


In [70]:
# JS local
print("JS promediada por fila entre atenciones")
for i, (bsattn, sattn) in enumerate(zip(bad_student_attentions, student_attentions)):
    print(js_divergence_attention(bsattn, sattn))

JS promediada por fila entre atenciones
tensor(0.0033)
tensor(0.0310)
tensor(0.0314)
tensor(0.0261)
tensor(0.0192)
tensor(0.0209)
tensor(0.0132)
tensor(0.0114)
tensor(0.0185)
tensor(0.0163)
tensor(0.0261)


# Bad Student - Student local comparison

In [71]:
# Cosine similarity entre attention maps
print("Similitud coseno entre atenciones")
for i, (bsattn, slattn) in enumerate(zip(bad_student_attentions, student_local_attentions)):
    print(cosine_similarity_tensor(bsattn, slattn))

Similitud coseno entre atenciones
tensor(0.9771)
tensor(0.8352)
tensor(0.8108)
tensor(0.8770)
tensor(0.9060)
tensor(0.9127)
tensor(0.9176)
tensor(0.9324)
tensor(0.9307)
tensor(0.8666)
tensor(0.8875)


In [72]:
# Frobenius distance entre attention maps
print("Distancia Frobenius entre atenciones")
for i, (bsattn, slattn) in enumerate(zip(bad_student_attentions, student_local_attentions)):
    print(frobenius_difference(bsattn, slattn))

Distancia Frobenius entre atenciones
tensor(1.1958)
tensor(3.9588)
tensor(3.9554)
tensor(2.9651)
tensor(2.5630)
tensor(2.4949)
tensor(2.4419)
tensor(2.1416)
tensor(2.1978)
tensor(3.2702)
tensor(2.9639)


In [73]:
# JS local
print("JS promediada por fila entre atenciones")
for i, (bsattn, slattn) in enumerate(zip(bad_student_attentions, student_local_attentions)):
    print(js_divergence_attention(bsattn, slattn))

JS promediada por fila entre atenciones
tensor(0.0046)
tensor(0.0336)
tensor(0.0320)
tensor(0.0200)
tensor(0.0129)
tensor(0.0132)
tensor(0.0149)
tensor(0.0141)
tensor(0.0148)
tensor(0.0256)
tensor(0.0244)


# Student - Student comparison

In [74]:
# Cosine similarity entre attention maps
print("Similitud coseno entre atenciones")
for i, (sattn, slattn) in enumerate(zip(student_attentions, student_local_attentions)):
    print(cosine_similarity_tensor(sattn, slattn))

Similitud coseno entre atenciones
tensor(0.9861)
tensor(0.9723)
tensor(0.9014)
tensor(0.8975)
tensor(0.9639)
tensor(0.9310)
tensor(0.9472)
tensor(0.9190)
tensor(0.9225)
tensor(0.9085)
tensor(0.9008)


In [75]:
# Frobenius distance entre attention maps
print("Distancia Frobenius entre atenciones")
for i, (sattn, slattn) in enumerate(zip(student_attentions, student_local_attentions)):
    print(frobenius_difference(sattn, slattn))

Distancia Frobenius entre atenciones
tensor(0.9286)
tensor(1.4254)
tensor(2.9167)
tensor(2.7888)
tensor(1.6122)
tensor(2.2272)
tensor(1.9708)
tensor(2.3450)
tensor(2.3063)
tensor(2.7393)
tensor(2.7906)


In [76]:
# JS local
print("JS promediada por fila entre atenciones")
for i, (sattn, slattn) in enumerate(zip(student_attentions, student_local_attentions)):
    print(js_divergence_attention(sattn, slattn))

JS promediada por fila entre atenciones
tensor(0.0029)
tensor(0.0059)
tensor(0.0123)
tensor(0.0104)
tensor(0.0088)
tensor(0.0195)
tensor(0.0130)
tensor(0.0161)
tensor(0.0174)
tensor(0.0196)
tensor(0.0192)


# Effective ranks comparisons

In [77]:
# Participation ratio
print("Participation ratio")
print("index | teacher_avg | teacher_roll | baseline | bad_student |student | student_local")
print("-----------------------------------------------")
for i, (tattn, t2attn, battn, bsattn, sattn, slattn) in enumerate(zip(teacher_avg_attentions, teacher_rollouts, baseline_attentions, bad_student_attentions, student_attentions, student_local_attentions)):
    print(f"{i} | {effective_rank_participation_ratio(tattn):4f} | {effective_rank_participation_ratio(t2attn):4f} | {effective_rank_participation_ratio(battn):4f} | {effective_rank_participation_ratio(bsattn):4f} | {effective_rank_participation_ratio(sattn):4f} | {effective_rank_participation_ratio(slattn):4f} ")


Participation ratio
index | teacher_avg | teacher_roll | baseline | bad_student |student | student_local
-----------------------------------------------
0 | 1.356875 | 67.457176 | 1.275496 | 1.291532 | 1.355018 | 1.348717 
1 | 1.005610 | 1.724615 | 1.605922 | 1.309881 | 1.570958 | 1.565603 
2 | 1.000082 | 1.265335 | 1.597238 | 1.426726 | 1.700024 | 1.977653 
3 | 1.000213 | 1.277876 | 1.883908 | 1.358773 | 1.882176 | 2.025190 
4 | 1.005978 | 1.622420 | 1.836384 | 1.360982 | 1.730386 | 1.842880 
5 | 1.002858 | 1.704638 | 2.638350 | 1.398054 | 1.836006 | 1.851146 
6 | 1.006170 | 1.995486 | 2.376256 | 1.455059 | 1.836463 | 1.976044 
7 | 1.007666 | 2.028602 | 3.335663 | 1.519558 | 1.753374 | 1.618317 
8 | 1.007667 | 1.945815 | 2.559661 | 1.465775 | 1.704589 | 1.566849 
9 | 1.007110 | 1.909445 | 4.253060 | 1.435512 | 1.747687 | 1.941132 
10 | 1.006790 | 1.885484 | 3.194835 | 1.545660 | 1.714019 | 1.744292 


In [78]:
# Entropy based effective rank
print("Entropy based effective rank")
print("index | teacher_avg | teacher_roll | baseline | bad_student |student | student_local")
print("-----------------------------------------------")
for i, (tattn, t2attn, battn, bsattn, sattn, slattn) in enumerate(zip(teacher_avg_attentions, teacher_rollouts, baseline_attentions, bad_student_attentions, student_attentions, student_local_attentions)):
    print(f"{i} | {effective_rank_entropy(tattn):4f} | {effective_rank_entropy(t2attn):4f} | {effective_rank_entropy(battn):4f} | {effective_rank_entropy(bsattn):4f} | {effective_rank_entropy(sattn):4f} | {effective_rank_entropy(slattn):4f} ")

Entropy based effective rank
index | teacher_avg | teacher_roll | baseline | bad_student |student | student_local
-----------------------------------------------
0 | 22.324389 | 2010.146240 | 11.610521 | 12.529155 | 14.208723 | 15.655766 
1 | 4.120300 | 1798.869263 | 28.762945 | 12.878530 | 20.299650 | 21.722729 
2 | 1.356920 | 1633.819702 | 27.222998 | 12.064932 | 27.272469 | 35.889946 
3 | 1.611693 | 1643.186646 | 37.570496 | 17.062733 | 28.054041 | 47.068680 
4 | 2.645257 | 1778.611450 | 39.992477 | 17.879768 | 42.169987 | 49.956593 
5 | 2.623758 | 1796.103394 | 49.555668 | 21.525631 | 44.914436 | 50.806606 
6 | 4.044635 | 1836.495728 | 55.765526 | 28.012730 | 45.987522 | 52.178757 
7 | 4.690934 | 1839.722046 | 58.389290 | 28.416212 | 41.186562 | 42.622803 
8 | 4.206147 | 1830.746948 | 58.099644 | 26.996763 | 39.166691 | 35.830936 
9 | 3.517950 | 1826.503052 | 58.180458 | 26.414501 | 39.683723 | 46.903019 
10 | 5.958082 | 1822.801880 | 62.297825 | 23.052923 | 34.785976 | 37.954002 


# Comparación con promedio y rollout total

In [79]:
teacher_total_avg_attention = sum(teacher_attentions)/len(teacher_attentions)

def total_rollout(attentions):
    I = torch.eye(2048, device="cpu")
    R = None
    for A in attentions:

        attn = A + I
        attn = attn / attn.sum(dim=-1, keepdim=True)

        if R is None:
            R = attn
        else:
            R = R @ attn
    
    return R

teacher_total_rollout = total_rollout(teacher_attentions)

In [80]:
baseline_total_avg_attention = sum(baseline_attentions)/len(baseline_attentions)
baseline_total_rollout = total_rollout(baseline_attentions)

In [81]:
student_total_avg_attention = sum(student_attentions)/len(student_attentions)
student_total_rollout = total_rollout(student_attentions)

In [82]:
student_local_total_avg_attention = sum(student_local_attentions)/len(student_local_attentions)
student_local_total_rollout = total_rollout(student_local_attentions)

In [83]:
bad_student_total_avg_attention = sum(bad_student_attentions)/len(bad_student_attentions)
bad_student_total_rollout = total_rollout(bad_student_attentions)

In [84]:
# Cosine similarity entre attention maps
print("Similitud coseno")

print("Promedio student - Promedio teacher")
print(frobenius_difference(student_total_avg_attention, teacher_total_avg_attention))

print("Promedio student - Rollout teacher")
print(cosine_similarity_tensor(student_total_avg_attention, teacher_total_rollout))

print("Promedio student local - Promedio teacher")
print(cosine_similarity_tensor(student_local_total_avg_attention, teacher_total_avg_attention))

print("Promedio student local - Rollout teacher")
print(cosine_similarity_tensor(student_local_total_avg_attention, teacher_total_rollout))

print("Rollout student - Promedio teacher")
print(cosine_similarity_tensor(student_total_rollout, teacher_total_avg_attention))

print("Rollout student - Rollout teacher")
print(cosine_similarity_tensor(student_total_rollout, teacher_total_rollout))

print("Rollout student local - Promedio teacher")
print(cosine_similarity_tensor(student_local_total_rollout, teacher_total_avg_attention))

print("Rollout student local - Rollout teacher")
print(cosine_similarity_tensor(student_local_total_rollout, teacher_total_rollout))

print("Promedio student - Promedio student local")
print(cosine_similarity_tensor(student_total_avg_attention, student_local_total_avg_attention))

print("Promedio student - Rollout student local")
print(cosine_similarity_tensor(student_total_avg_attention, student_local_total_rollout))

print("Rollout student - Promedio student local")
print(cosine_similarity_tensor(student_total_rollout, student_local_total_avg_attention))

print("Rollout student - Rollout student local")
print(cosine_similarity_tensor(student_total_rollout, student_local_total_rollout))

print("Baseline promedio - Promedio teacher")
print(cosine_similarity_tensor(baseline_total_avg_attention, teacher_total_avg_attention))

print("Baseline promedio - Rollout teacher")
print(cosine_similarity_tensor(baseline_total_avg_attention, teacher_total_rollout))

print("Baseline promedio - Promedio student")
print(cosine_similarity_tensor(baseline_total_avg_attention, student_total_avg_attention))

print("Baseline promedio - Rollout student")
print(cosine_similarity_tensor(baseline_total_avg_attention, student_total_rollout))

print("Baseline promedio - Promedio student local")
print(cosine_similarity_tensor(baseline_total_avg_attention, student_local_total_avg_attention))

print("Baseline promedio - Rollout student local")
print(cosine_similarity_tensor(baseline_total_avg_attention, student_local_total_rollout))

print("Baseline promedio - Promedio bad student")
print(cosine_similarity_tensor(baseline_total_avg_attention, bad_student_total_avg_attention))

print("Baseline promedio - Rollout bad student")
print(cosine_similarity_tensor(baseline_total_avg_attention, bad_student_total_rollout))

print("Baseline rollout - Promedio teacher")
print(cosine_similarity_tensor(baseline_total_rollout, teacher_total_avg_attention))

print("Baseline rollout - Rollout teacher")
print(cosine_similarity_tensor(baseline_total_rollout, teacher_total_rollout))

print("Baseline rollout - Promedio student")
print(cosine_similarity_tensor(baseline_total_rollout, student_total_avg_attention))

print("Baseline rollout - Rollout student")
print(cosine_similarity_tensor(baseline_total_rollout, student_total_rollout))

print("Baseline rollout - Promedio student local")
print(cosine_similarity_tensor(baseline_total_rollout, student_local_total_avg_attention))

print("Baseline rollout - Rollout student local")
print(cosine_similarity_tensor(baseline_total_rollout, student_local_total_rollout))

print("Baseline rollout - Promedio bad student")
print(cosine_similarity_tensor(baseline_total_rollout, bad_student_total_avg_attention))

print("Baseline rollout - Rollout bad student")
print(cosine_similarity_tensor(baseline_total_rollout, bad_student_total_rollout))

print("Promedio bad student - Promedio teacher")
print(cosine_similarity_tensor(bad_student_total_avg_attention, teacher_total_avg_attention))

print("Promedio bad student - Rollout teacher")
print(cosine_similarity_tensor(bad_student_total_avg_attention, teacher_total_rollout))

print("Promedio bad student - Promedio student")
print(cosine_similarity_tensor(bad_student_total_avg_attention, student_total_avg_attention))

print("Promedio bad student - Rollout student")
print(cosine_similarity_tensor(bad_student_total_avg_attention, student_total_rollout))

print("Promedio bad student - Promedio student local")
print(cosine_similarity_tensor(bad_student_total_avg_attention, student_local_total_avg_attention))

print("Promedio bad student - Rollout student local")
print(cosine_similarity_tensor(bad_student_total_avg_attention, student_local_total_rollout))

print("Rollout bad student - Promedio teacher")
print(cosine_similarity_tensor(bad_student_total_rollout, teacher_total_avg_attention))

print("Rollout bad student - Rollout teacher")
print(cosine_similarity_tensor(bad_student_total_rollout, teacher_total_rollout))

print("Rollout bad student - Promedio student")
print(cosine_similarity_tensor(bad_student_total_rollout, student_total_avg_attention))

print("Rollout bad student - Rollout student")
print(cosine_similarity_tensor(bad_student_total_rollout, student_total_rollout))

print("Rollout bad student - Promedio student local")
print(cosine_similarity_tensor(bad_student_total_rollout, student_local_total_avg_attention))

print("Rollout bad student - Rollout student local")
print(cosine_similarity_tensor(bad_student_total_rollout, student_local_total_rollout))

Similitud coseno
Promedio student - Promedio teacher
tensor(21.2893)
Promedio student - Rollout teacher
tensor(0.1915)
Promedio student local - Promedio teacher
tensor(0.3312)
Promedio student local - Rollout teacher
tensor(0.2258)
Rollout student - Promedio teacher
tensor(0.9794)
Rollout student - Rollout teacher
tensor(0.9808)
Rollout student local - Promedio teacher
tensor(0.9754)
Rollout student local - Rollout teacher
tensor(0.9747)
Promedio student - Promedio student local
tensor(0.9843)
Promedio student - Rollout student local
tensor(0.3016)
Rollout student - Promedio student local
tensor(0.3252)
Rollout student - Rollout student local
tensor(0.9991)
Baseline promedio - Promedio teacher
tensor(0.4119)
Baseline promedio - Rollout teacher
tensor(0.3228)
Baseline promedio - Promedio student
tensor(0.8599)
Baseline promedio - Rollout student
tensor(0.4162)
Baseline promedio - Promedio student local
tensor(0.8638)
Baseline promedio - Rollout student local
tensor(0.4434)
Baseline prom

In [85]:
# Distancia Frobenius entre attention maps
print("Distancia Frobenius")

print("Promedio student - Promedio teacher")
print(frobenius_difference(student_total_avg_attention, teacher_total_avg_attention))

print("Promedio student - Rollout teacher")
print(frobenius_difference(student_total_avg_attention, teacher_total_rollout))

print("Promedio student local - Promedio teacher")
print(frobenius_difference(student_local_total_avg_attention, teacher_total_avg_attention))

print("Promedio student local - Rollout teacher")
print(frobenius_difference(student_local_total_avg_attention, teacher_total_rollout))

print("Rollout student - Promedio teacher")
print(frobenius_difference(student_total_rollout, teacher_total_avg_attention))

print("Rollout student - Rollout teacher")
print(frobenius_difference(student_total_rollout, teacher_total_rollout))

print("Rollout student local - Promedio teacher")
print(frobenius_difference(student_local_total_rollout, teacher_total_avg_attention))

print("Rollout student local - Rollout teacher")
print(frobenius_difference(student_local_total_rollout, teacher_total_rollout))

print("Promedio student - Promedio student local")
print(frobenius_difference(student_total_avg_attention, student_local_total_avg_attention))

print("Promedio student - Rollout student local")
print(frobenius_difference(student_total_avg_attention, student_local_total_rollout))

print("Rollout student - Promedio student local")
print(frobenius_difference(student_total_rollout, student_local_total_avg_attention))

print("Rollout student - Rollout student local")
print(frobenius_difference(student_total_rollout, student_local_total_rollout))

print("Baseline promedio - Promedio teacher")
print(frobenius_difference(baseline_total_avg_attention, teacher_total_avg_attention))

print("Baseline promedio - Rollout teacher")
print(frobenius_difference(baseline_total_avg_attention, teacher_total_rollout))

print("Baseline promedio - Promedio student")
print(frobenius_difference(baseline_total_avg_attention, student_total_avg_attention))

print("Baseline promedio - Rollout student")
print(frobenius_difference(baseline_total_avg_attention, student_total_rollout))

print("Baseline promedio - Promedio student local")
print(frobenius_difference(baseline_total_avg_attention, student_local_total_avg_attention))

print("Baseline promedio - Rollout student local")
print(frobenius_difference(baseline_total_avg_attention, student_local_total_rollout))

print("Baseline promedio - Promedio bad student")
print(frobenius_difference(baseline_total_avg_attention, bad_student_total_avg_attention))

print("Baseline promedio - Rollout bad student")
print(frobenius_difference(baseline_total_avg_attention, bad_student_total_rollout))

print("Baseline rollout - Promedio teacher")
print(frobenius_difference(baseline_total_rollout, teacher_total_avg_attention))

print("Baseline rollout - Rollout teacher")
print(frobenius_difference(baseline_total_rollout, teacher_total_rollout))

print("Baseline rollout - Promedio student")
print(frobenius_difference(baseline_total_rollout, student_total_avg_attention))

print("Baseline rollout - Rollout student")
print(frobenius_difference(baseline_total_rollout, student_total_rollout))

print("Baseline rollout - Promedio student local")
print(frobenius_difference(baseline_total_rollout, student_local_total_avg_attention))

print("Baseline rollout - Rollout student local")
print(frobenius_difference(baseline_total_rollout, student_local_total_rollout))

print("Baseline rollout - Promedio bad student")
print(frobenius_difference(baseline_total_rollout, bad_student_total_avg_attention))

print("Baseline rollout - Rollout bad student")
print(frobenius_difference(baseline_total_rollout, bad_student_total_rollout))

print("Promedio bad student - Promedio teacher")
print(frobenius_difference(bad_student_total_avg_attention, teacher_total_avg_attention))

print("Promedio bad student - Rollout teacher")
print(frobenius_difference(bad_student_total_avg_attention, teacher_total_rollout))

print("Promedio bad student - Promedio student")
print(frobenius_difference(bad_student_total_avg_attention, student_total_avg_attention))

print("Promedio bad student - Rollout student")
print(frobenius_difference(bad_student_total_avg_attention, student_total_rollout))

print("Promedio bad student - Promedio student local")
print(frobenius_difference(bad_student_total_avg_attention, student_local_total_avg_attention))

print("Promedio bad student - Rollout student local")
print(frobenius_difference(bad_student_total_avg_attention, student_local_total_rollout))

print("Rollout bad student - Promedio teacher")
print(frobenius_difference(bad_student_total_rollout, teacher_total_avg_attention))

print("Rollout bad student - Rollout teacher")
print(frobenius_difference(bad_student_total_rollout, teacher_total_rollout))

print("Rollout bad student - Promedio student")
print(frobenius_difference(bad_student_total_rollout, student_total_avg_attention))

print("Rollout bad student - Rollout student")
print(frobenius_difference(bad_student_total_rollout, student_total_rollout))

print("Rollout bad student - Promedio student local")
print(frobenius_difference(bad_student_total_rollout, student_local_total_avg_attention))

print("Rollout bad student - Rollout student local")
print(frobenius_difference(bad_student_total_rollout, student_local_total_rollout))

Distancia Frobenius
Promedio student - Promedio teacher
tensor(21.2893)
Promedio student - Rollout teacher
tensor(44.5205)
Promedio student local - Promedio teacher
tensor(21.1293)
Promedio student local - Rollout teacher
tensor(44.3075)
Rollout student - Promedio teacher
tensor(7.5341)
Rollout student - Rollout teacher
tensor(18.6977)
Rollout student local - Promedio teacher
tensor(5.6995)
Rollout student local - Rollout teacher
tensor(21.9213)
Promedio student - Promedio student local
tensor(1.0189)
Promedio student - Rollout student local
tensor(23.5821)
Rollout student - Promedio student local
tensor(26.5897)
Rollout student - Rollout student local
tensor(3.4213)
Baseline promedio - Promedio teacher
tensor(20.5712)
Baseline promedio - Rollout teacher
tensor(43.6570)
Baseline promedio - Promedio student
tensor(3.1475)
Baseline promedio - Rollout student
tensor(25.9495)
Baseline promedio - Promedio student local
tensor(3.1252)
Baseline promedio - Rollout student local
tensor(22.6139)

In [86]:
# JS local entre attention maps
print("JS local")

print("Promedio student - Promedio teacher")
print(js_divergence_attention(student_total_avg_attention, teacher_total_avg_attention))

print("Promedio student - Rollout teacher")
print(js_divergence_attention(student_total_avg_attention, teacher_total_rollout))

print("Promedio student local - Promedio teacher")
print(js_divergence_attention(student_local_total_avg_attention, teacher_total_avg_attention))

print("Promedio student local - Rollout teacher")
print(js_divergence_attention(student_local_total_avg_attention, teacher_total_rollout))

print("Rollout student - Promedio teacher")
print(js_divergence_attention(student_total_rollout, teacher_total_avg_attention))

print("Rollout student - Rollout teacher")
print(js_divergence_attention(student_total_rollout, teacher_total_rollout))

print("Rollout student local - Promedio teacher")
print(js_divergence_attention(student_local_total_rollout, teacher_total_avg_attention))

print("Rollout student local - Rollout teacher")
print(js_divergence_attention(student_local_total_rollout, teacher_total_rollout))

print("Promedio student - Promedio student local")
print(js_divergence_attention(student_total_avg_attention, student_local_total_avg_attention))

print("Promedio student - Rollout student local")
print(js_divergence_attention(student_total_avg_attention, student_local_total_rollout))

print("Rollout student - Promedio student local")
print(js_divergence_attention(student_total_rollout, student_local_total_avg_attention))

print("Rollout student - Rollout student local")
print(js_divergence_attention(student_total_rollout, student_local_total_rollout))

print("Baseline promedio - Promedio teacher")
print(js_divergence_attention(baseline_total_avg_attention, teacher_total_avg_attention))

print("Baseline promedio - Rollout teacher")
print(js_divergence_attention(baseline_total_avg_attention, teacher_total_rollout))

print("Baseline promedio - Promedio student")
print(js_divergence_attention(baseline_total_avg_attention, student_total_avg_attention))

print("Baseline promedio - Rollout student")
print(js_divergence_attention(baseline_total_avg_attention, student_total_rollout))

print("Baseline promedio - Promedio student local")
print(js_divergence_attention(baseline_total_avg_attention, student_local_total_avg_attention))

print("Baseline promedio - Rollout student local")
print(js_divergence_attention(baseline_total_avg_attention, student_local_total_rollout))

print("Baseline promedio - Promedio bad student")
print(js_divergence_attention(baseline_total_avg_attention, bad_student_total_avg_attention))

print("Baseline promedio - Rollout bad student")
print(js_divergence_attention(baseline_total_avg_attention, bad_student_total_rollout))

print("Baseline rollout - Promedio teacher")
print(js_divergence_attention(baseline_total_rollout, teacher_total_avg_attention))

print("Baseline rollout - Rollout teacher")
print(js_divergence_attention(baseline_total_rollout, teacher_total_rollout))

print("Baseline rollout - Promedio student")
print(js_divergence_attention(baseline_total_rollout, student_total_avg_attention))

print("Baseline rollout - Rollout student")
print(js_divergence_attention(baseline_total_rollout, student_total_rollout))

print("Baseline rollout - Promedio student local")
print(js_divergence_attention(baseline_total_rollout, student_local_total_avg_attention))

print("Baseline rollout - Rollout student local")
print(js_divergence_attention(baseline_total_rollout, student_local_total_rollout))

print("Baseline rollout - Promedio bad student")
print(js_divergence_attention(baseline_total_rollout, bad_student_total_avg_attention))

print("Baseline rollout - Rollout bad student")
print(js_divergence_attention(baseline_total_rollout, bad_student_total_rollout))

print("Promedio bad student - Promedio teacher")
print(js_divergence_attention(bad_student_total_avg_attention, teacher_total_avg_attention))

print("Promedio bad student - Rollout teacher")
print(js_divergence_attention(bad_student_total_avg_attention, teacher_total_rollout))

print("Promedio bad student - Promedio student")
print(js_divergence_attention(bad_student_total_avg_attention, student_total_avg_attention))

print("Promedio bad student - Rollout student")
print(js_divergence_attention(bad_student_total_avg_attention, student_total_rollout))

print("Promedio bad student - Promedio student local")
print(js_divergence_attention(bad_student_total_avg_attention, student_local_total_avg_attention))

print("Promedio bad student - Rollout student local")
print(js_divergence_attention(bad_student_total_avg_attention, student_local_total_rollout))

print("Rollout bad student - Promedio teacher")
print(js_divergence_attention(bad_student_total_rollout, teacher_total_avg_attention))

print("Rollout bad student - Rollout teacher")
print(js_divergence_attention(bad_student_total_rollout, teacher_total_rollout))

print("Rollout bad student - Promedio student")
print(js_divergence_attention(bad_student_total_rollout, student_total_avg_attention))

print("Rollout bad student - Rollout student")
print(js_divergence_attention(bad_student_total_rollout, student_total_rollout))

print("Rollout bad student - Promedio student local")
print(js_divergence_attention(bad_student_total_rollout, student_local_total_avg_attention))

print("Rollout bad student - Rollout student local")
print(js_divergence_attention(bad_student_total_rollout, student_local_total_rollout))

JS local
Promedio student - Promedio teacher
tensor(0.1890)
Promedio student - Rollout teacher
tensor(0.6408)
Promedio student local - Promedio teacher
tensor(0.1878)
Promedio student local - Rollout teacher
tensor(0.6303)
Rollout student - Promedio teacher
tensor(0.1451)
Rollout student - Rollout teacher
tensor(0.1615)
Rollout student local - Promedio teacher
tensor(0.1500)
Rollout student local - Rollout teacher
tensor(0.1989)
Promedio student - Promedio student local
tensor(0.0027)
Promedio student - Rollout student local
tensor(0.3165)
Rollout student - Promedio student local
tensor(0.3270)
Rollout student - Rollout student local
tensor(0.0039)
Baseline promedio - Promedio teacher
tensor(0.2088)
Baseline promedio - Rollout teacher
tensor(0.6043)
Baseline promedio - Promedio student
tensor(0.0379)
Baseline promedio - Rollout student
tensor(0.2894)
Baseline promedio - Promedio student local
tensor(0.0320)
Baseline promedio - Rollout student local
tensor(0.2492)
Baseline promedio - Pr

In [87]:
# Participation ratio
print("Participation ratio")
print("Teacher avg")
print(effective_rank_participation_ratio(teacher_total_avg_attention))
print("Student avg")
print(effective_rank_participation_ratio(student_total_avg_attention))
print("Student local avg")
print(effective_rank_participation_ratio(student_local_total_avg_attention))
print("Teacher rollout")
print(effective_rank_participation_ratio(teacher_total_rollout))
print("Student rollout")
print(effective_rank_participation_ratio(student_total_rollout))
print("Student local rollout")
print(effective_rank_participation_ratio(student_local_total_rollout))
print("Baseline avg")
print(effective_rank_participation_ratio(baseline_total_avg_attention))
print("Baseline avg")
print(effective_rank_participation_ratio(baseline_total_rollout))
print("Bad student avg")
print(effective_rank_participation_ratio(bad_student_total_avg_attention))
print("Bad student rollout")
print(effective_rank_participation_ratio(bad_student_total_rollout))

Participation ratio
Teacher avg
tensor(1.0027)
Student avg
tensor(1.5276)
Student local avg
tensor(1.5470)
Teacher rollout
tensor(1.)
Student rollout
tensor(1.0005)
Student local rollout
tensor(1.0009)
Baseline avg
tensor(1.5052)
Baseline avg
tensor(1.0002)
Bad student avg
tensor(1.3187)
Bad student rollout
tensor(1.0004)


In [88]:
# Entropy based effective rank
print("Entropy based effective rank")
print("Teacher avg")
print(effective_rank_entropy(teacher_total_avg_attention))
print("Student avg")
print(effective_rank_entropy(student_total_avg_attention))
print("Student local avg")
print(effective_rank_entropy(student_local_total_avg_attention))
print("Teacher rollout")
print(effective_rank_entropy(teacher_total_rollout))
print("Student rollout")
print(effective_rank_entropy(student_total_rollout))
print("Student local rollout")
print(effective_rank_entropy(student_local_total_rollout))
print("Baseline avg")
print(effective_rank_entropy(baseline_total_avg_attention))
print("Baseline avg")
print(effective_rank_entropy(baseline_total_rollout))
print("Bad student avg")
print(effective_rank_entropy(bad_student_total_avg_attention))
print("Bad student rollout")
print(effective_rank_entropy(bad_student_total_rollout))

Entropy based effective rank
Teacher avg
tensor(3.2783)
Student avg
tensor(34.8996)
Student local avg
tensor(41.9092)
Teacher rollout
tensor(1.0003)
Student rollout
tensor(1.7412)
Student local rollout
tensor(1.9371)
Baseline avg
tensor(31.2490)
Baseline avg
tensor(1.5520)
Bad student avg
tensor(18.5942)
Bad student rollout
tensor(1.5935)


# B. Layer-wise divergence profiles

In [ ]:
import os; os.makedirs("../images", exist_ok=True)
# Layer-wise divergence: each student vs teacher_avg_attentions (per layer)
STUDENT_KEYS = ["baseline", "bad_student", "student", "student_local"]
MODEL_COLORS = {
    "baseline":      "tab:blue",
    "bad_student":   "tab:red",
    "student":       "tab:green",
    "student_local": "tab:purple",
}

layer_idx = list(range(11))

cosine_by_model    = {}
frobenius_by_model = {}
js_by_model        = {}

for name in STUDENT_KEYS:
    s_attns = model_attentions[name]
    cos_v, frob_v, js_v = [], [], []
    for k in range(11):
        t = teacher_avg_attentions[k]
        s = s_attns[k]
        cos_v.append(cosine_similarity_tensor(t, s).item())
        frob_v.append(frobenius_difference(t, s).item())
        js_v.append(js_divergence_attention(t, s).item())
    cosine_by_model[name]    = cos_v
    frobenius_by_model[name] = frob_v
    js_by_model[name]        = js_v

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (data, title) in zip(axes, [
    (cosine_by_model,    "Cosine similarity"),
    (frobenius_by_model, "Frobenius distance"),
    (js_by_model,        "JS divergence"),
]):
    for name in STUDENT_KEYS:
        ax.plot(layer_idx, data[name], label=MODEL_LABELS[name],
                color=MODEL_COLORS[name], marker="o", linewidth=2, markersize=5)
    ax.set_xlabel("Layer index (student)")
    ax.set_title(title)
    ax.set_xticks(layer_idx)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
plt.suptitle("Layer-wise divergence profiles (teacher_avg vs student)", y=1.02)
plt.tight_layout()
plt.savefig("../images/agnews_layer_divergence_profiles.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: ../images/agnews_layer_divergence_profiles.png")


# C. Per-head similarity heatmaps

In [ ]:
# Compute per-head attention patterns (N=50, max_seq=128)
eval_prompts_50 = [build_prompt(t) for t in eval_texts[:50]]

head_patterns = {}
for name, model in MODELS.items():
    print(f"Head patterns for {MODEL_LABELS[name]}...")
    model.to(device)
    head_patterns[name] = compute_head_avg_patterns(
        model, eval_prompts_50, tokenizer, device, n=50, max_seq=128
    )
    model.cpu()
    torch.cuda.empty_cache()

print("Done. Shape:", head_patterns["teacher"].shape)


In [ ]:
import os; os.makedirs("../images", exist_ok=True)
# Per-head cosine-similarity heatmap: teacher vs each student
# 32x32 matrix averaged over 11 matched layer pairs
teacher_hp = head_patterns["teacher"]   # [22, n_heads, 128]
n_heads    = teacher_hp.shape[1]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, name in zip(axes, STUDENT_KEYS):
    student_hp = head_patterns[name]    # [11, n_heads, 128]
    sim_sum = torch.zeros(n_heads, n_heads)
    for k in range(11):
        t_paired = (teacher_hp[2*k] + teacher_hp[2*k+1]) / 2
        s        = student_hp[k]
        t_norm   = t_paired / (t_paired.norm(dim=1, keepdim=True) + 1e-12)
        s_norm   = s        / (s.norm(dim=1, keepdim=True) + 1e-12)
        sim_sum += (t_norm @ s_norm.T)
    sim_avg = (sim_sum / 11).numpy()
    im = ax.imshow(sim_avg, vmin=-1, vmax=1, cmap="RdBu_r", aspect="auto")
    ax.set_title(f"Teacher vs {MODEL_LABELS[name]}")
    ax.set_xlabel("Student head")
    ax.set_ylabel("Teacher head")
    plt.colorbar(im, ax=ax, shrink=0.8)
plt.suptitle("Per-head cosine similarity (avg over 11 matched layer pairs, N=50)", y=1.02)
plt.tight_layout()
plt.savefig("../images/agnews_head_similarity_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: ../images/agnews_head_similarity_heatmap.png")


# D. Hidden states — Linear CKA

In [ ]:
# Compute hidden states for all models (N=100, max_seq=512)
eval_prompts_100 = [build_prompt(t) for t in eval_texts[:100]]

hidden_states_all = {}
for name, model in MODELS.items():
    print(f"Hidden states for {MODEL_LABELS[name]}...")
    model.to(device)
    hidden_states_all[name] = compute_hidden_states(
        model, eval_prompts_100, tokenizer, device, n=100, max_seq=512
    )
    model.cpu()
    torch.cuda.empty_cache()

print("Shapes:", {k: tuple(v.shape) for k, v in hidden_states_all.items()})


In [ ]:
import os; os.makedirs("../images", exist_ok=True)
# Linear CKA heatmap: teacher layers vs student layers (N=100)
import numpy as np

teacher_hs = hidden_states_all["teacher"]  # [23, 100, 2048]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, name in zip(axes, STUDENT_KEYS):
    student_hs = hidden_states_all[name]   # [12, 100, 2048]
    n_t = teacher_hs.shape[0]
    n_s = student_hs.shape[0]
    cka_mat = np.zeros((n_t, n_s))
    for i in range(n_t):
        for j in range(n_s):
            cka_mat[i, j] = linear_cka(teacher_hs[i], student_hs[j])
    im = ax.imshow(cka_mat, vmin=0, vmax=1, cmap="Blues", aspect="auto")
    ax.set_title(f"Teacher vs {MODEL_LABELS[name]}")
    ax.set_xlabel("Student layer")
    ax.set_ylabel("Teacher layer")
    plt.colorbar(im, ax=ax, shrink=0.8)
plt.suptitle("Linear CKA: teacher vs student hidden states (N=100)", y=1.02)
plt.tight_layout()
plt.savefig("../images/agnews_cka_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: ../images/agnews_cka_heatmap.png")
